# SurvFace Grad-CAM — 05. Saliency–compression association

saliency를 geometry와 retrieval에 각각 결합하며 registered와 unmated, protocol, threshold policy를 서로 pooling하지 않습니다.

이 노트북은 한 단계만 실행하는 thin runbook입니다. 계산 구현은 `research/experiments/step4_workflow.py`에 있습니다.

이 단계는 GPU 연산이 아니라 대용량 CSV join과 identity-cluster bootstrap을 수행하는 CPU 단계입니다. 입력을 100,000행씩 strict join하고, association에 필요한 열만 임시 Parquet로 투영한 뒤 500회 weighted-rerank bootstrap을 실행합니다. 반복 표본을 대형 DataFrame으로 펼치지 않지만 각 반복의 평균 rank는 identity multiplicity로 정확히 재계산합니다. 모든 결과가 준비되기 전에는 최종 경로에 publish하지 않으므로 중단 시 임시 파일만 정리됩니다. 실행 중에는 10% milestone과 60초 heartbeat가 표시됩니다.

이전 `A001` 실패 manifest는 보존됩니다. 개선된 코드를 commit하고 커널을 재시작한 뒤 이 노트북을 다시 실행하면 같은 run의 새 phase attempt로 기록됩니다.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
EXECUTION = CONFIG["execution"]
MODEL_PROFILE = str(EXECUTION["model_profile"])
MODE = str(EXECUTION["mode"])
DATA_FRACTION = float(EXECUTION["data_fraction"])
EXECUTE_STAGE = bool(EXECUTION["execute_stage"])
WRITE_OUTPUTS = bool(EXECUTION["write_outputs"])
OVERWRITE = bool(EXECUTION["overwrite"])
DATASET_ID = "survface"
ASSOCIATION = CONFIG["joint_analysis"]["association"]
BOOTSTRAP_REPEATS = int(ASSOCIATION["bootstrap_repeats"])

if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")
if EXECUTE_STAGE and not WRITE_OUTPUTS:
    raise ValueError("정식 단계 실행은 WRITE_OUTPUTS=True여야 합니다.")
if BOOTSTRAP_REPEATS < 0:
    raise ValueError("bootstrap_repeats는 0 이상이어야 합니다.")

from research.experiments import analyze_step4_saliency_compression
from research.runtime import ProgressReporter

PROGRESS = ProgressReporter(
    "SurvFace 05 saliency-compression",
    heartbeat_seconds=60,
    milestone_percent=10,
)
execution_plan = {
    "dataset_id": DATASET_ID,
    "device": "cpu",
    "join_mode": "streaming_100000_rows",
    "bootstrap_method": ASSOCIATION["bootstrap_method"],
    "bootstrap_repeats": BOOTSTRAP_REPEATS,
    "bootstrap_batch_size": 4,
    "bootstrap_rank_strategy": "weighted_rerank",
}
execution_plan


In [ ]:
if EXECUTE_STAGE:
    with PROGRESS.step(
        "streamed join and association",
        expected=f"{BOOTSTRAP_REPEATS} identity-cluster repeats on CPU",
    ):
        result = analyze_step4_saliency_compression(
            CONFIG_PATH,
            project_root=PROJECT_ROOT,
            dataset_id=DATASET_ID,
            execution_acknowledged=True,
            progress=PROGRESS.callback(key_prefix="step4-05:"),
        )
else:
    result = {
        "dataset_id": DATASET_ID,
        "status": "not_executed",
        "reason": "CONFIG execution gates are closed",
    }

result


## 다음 단계

`result["association_algorithm_version"]`과 새 phase attempt의 `status=completed`를 확인한 뒤 다음으로 진행합니다. 다음은 `04_representative_case_visualization.ipynb`입니다.

커널을 재시작한 뒤 다음 노트북을 위에서 아래로 실행합니다.